In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import random
import math
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression

In [2]:
train_df = pd.read_csv("train.csv")
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43957 entries, 0 to 43956
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              43957 non-null  int64 
 1   workclass        41459 non-null  object
 2   fnlwgt           43957 non-null  int64 
 3   education        43957 non-null  object
 4   educational-num  43957 non-null  int64 
 5   marital-status   43957 non-null  object
 6   occupation       41451 non-null  object
 7   relationship     43957 non-null  object
 8   race             43957 non-null  object
 9   gender           43957 non-null  object
 10  capital-gain     43957 non-null  int64 
 11  capital-loss     43957 non-null  int64 
 12  hours-per-week   43957 non-null  int64 
 13  native-country   43194 non-null  object
 14  income_>50K      43957 non-null  int64 
dtypes: int64(7), object(8)
memory usage: 5.0+ MB


In [3]:
train_df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income_>50K
0,67,Private,366425,Doctorate,16,Divorced,Exec-managerial,Not-in-family,White,Male,99999,0,60,United-States,1
1,17,Private,244602,12th,8,Never-married,Other-service,Own-child,White,Male,0,0,15,United-States,0
2,31,Private,174201,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,1
3,58,State-gov,110199,7th-8th,4,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States,0
4,25,State-gov,149248,Some-college,10,Never-married,Other-service,Not-in-family,Black,Male,0,0,40,United-States,0


In [4]:
train_df.describe()

,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week,income_>50K
count,43957.000000,4.395700e+04,43957.000000,43957.000000,43957.000000,43957.000000,43957.000000
mean,38.617149,1.896730e+05,10.074118,1093.559797,88.246491,40.407694,0.239279
std,13.734401,1.058215e+05,2.575092,7570.536063,404.588410,12.400303,0.426648
min,17.000000,1.349200e+04,1.000000,0.000000,0.000000,1.000000,0.000000
25%,28.000000,1.174960e+05,9.000000,0.000000,0.000000,40.000000,0.000000
50%,37.000000,1.781000e+05,10.000000,0.000000,0.000000,40.000000,0.000000
75%,48.000000,2.376710e+05,12.000000,0.000000,0.000000,45.000000,0.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000,1.000000


In [5]:
test_df = pd.read_csv("test.csv")
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899 entries, 0 to 898
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              899 non-null    int64 
 1   workclass        899 non-null    object
 2   fnlwgt           899 non-null    int64 
 3   education        899 non-null    object
 4   educational-num  899 non-null    int64 
 5   marital-status   899 non-null    object
 6   occupation       899 non-null    object
 7   relationship     899 non-null    object
 8   race             899 non-null    object
 9   gender           899 non-null    object
 10  capital-gain     899 non-null    int64 
 11  capital-loss     899 non-null    int64 
 12  hours-per-week   899 non-null    int64 
 13  native-country   899 non-null    object
dtypes: int64(6), object(8)
memory usage: 98.5+ KB


In [6]:
test_df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
0,39,Self-emp-not-inc,327120,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,Portugal
1,32,Private,123253,Assoc-acdm,12,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,42,United-States
2,47,Private,232628,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,Black,Male,0,0,40,United-States
3,19,Private,374262,12th,8,Never-married,Handlers-cleaners,Own-child,White,Male,0,0,20,United-States
4,46,Self-emp-not-inc,311231,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,40,United-States


In [7]:
# filter native country for only US results
# get rid of capital gain, capital loss, and fnlwgt columns
# fill any null values with "unknown" (safety net, although there do not seem to be any missing)
def clean_data(df):
    df = df[df["native-country"] == "United-States"]
    
    df = df.drop(columns=["fnlwgt", "capital-gain", "capital-loss"])
    
    df["workclass"] = df["workclass"].fillna("Unknown")
    df["occupation"] = df["occupation"].fillna("Unknown")
    
    return df

train_df = clean_data(train_df)
test_df = clean_data(test_df)


In [8]:
train_df.head()

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,hours-per-week,native-country,income_>50K
0,67,Private,Doctorate,16,Divorced,Exec-managerial,Not-in-family,White,Male,60,United-States,1
1,17,Private,12th,8,Never-married,Other-service,Own-child,White,Male,15,United-States,0
2,31,Private,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,40,United-States,1
3,58,State-gov,7th-8th,4,Married-civ-spouse,Transport-moving,Husband,White,Male,40,United-States,0
4,25,State-gov,Some-college,10,Never-married,Other-service,Not-in-family,Black,Male,40,United-States,0


In [9]:
test_df.head()

,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,hours-per-week,native-country
1,32,Private,Assoc-acdm,12,Married-civ-spouse,Craft-repair,Husband,White,Male,42,United-States
2,47,Private,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,Black,Male,40,United-States
3,19,Private,12th,8,Never-married,Handlers-cleaners,Own-child,White,Male,20,United-States
4,46,Self-emp-not-inc,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,40,United-States
5,45,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,50,United-States


In [10]:
# encoding 
train_df_encoding = pd.get_dummies(train_df, columns= ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country'], dtype=int)
test_df_encoding = pd.get_dummies(test_df, columns= ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country'], dtype=int)
train_df_encoding.head()
test_df_encoding.head()
train_df_encoding.info()
test_df_encoding.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39429 entries, 0 to 43956
Data columns (total 65 columns):
 #   Column                                Non-Null Count  Dtype
---  ------                                --------------  -----
 0   age                                   39429 non-null  int64
 1   educational-num                       39429 non-null  int64
 2   hours-per-week                        39429 non-null  int64
 3   income_>50K                           39429 non-null  int64
 4   workclass_Federal-gov                 39429 non-null  int64
 5   workclass_Local-gov                   39429 non-null  int64
 6   workclass_Never-worked                39429 non-null  int64
 7   workclass_Private                     39429 non-null  int64
 8   workclass_Self-emp-inc                39429 non-null  int64
 9   workclass_Self-emp-not-inc            39429 non-null  int64
 10  workclass_State-gov                   39429 non-null  int64
 11  workclass_Unknown                     39429 no

In [11]:
train_df_encoding, test_df_encoding = train_df_encoding.align(
    test_df_encoding, join='left', axis=1, fill_value=0)

In [12]:
set(train_df_encoding.columns)==set(test_df_encoding.columns)

True

In [13]:
numeric_cols = ['age', 'hours-per-week']
scaler = StandardScaler()
# fitting the scaler only to the train data
scaler.fit(train_df[numeric_cols])
# transforming both train and test
train_df_encoding[numeric_cols] = scaler.transform(train_df_encoding[numeric_cols])
test_df_encoding[numeric_cols] = scaler.transform(test_df_encoding[numeric_cols])
# print
print("Scaled train head:", train_df_encoding[numeric_cols].head())
print("Scaled train describe:", train_df_encoding[numeric_cols].describe())

Scaled train head:         age  hours-per-week
0  2.048918        1.568702
1 -1.568306       -2.037764
2 -0.555483       -0.034172
3  1.397817       -0.034172
4 -0.989550       -0.034172
Scaled train describe:                 age  hours-per-week
count  3.942900e+04    3.942900e+04
mean  -1.083051e-16   -3.243747e-17
std    1.000013e+00    1.000013e+00
min   -1.568306e+00   -3.159776e+00
25%   -7.725164e-01   -3.417220e-02
50%   -1.214163e-01   -3.417220e-02
75%    6.743728e-01    3.665463e-01
max    3.712840e+00    4.694306e+00


In [17]:
# defining features and target for education
EDU_FEATURES = [col for col in train_df_encoding.columns
                if any(col.startswith(p) for p in
                      ['age', 'workclass', 'occupation', 'income',
                'marital', 'relationship', 'race', 'gender', 'hours'])]
EDU_TARGET = 'educational-num'

# splitting the data
X_edu = train_df_encoding[EDU_FEATURES]
y_edu = train_df_encoding[EDU_TARGET]

X_edu_train, X_edu_dev, y_edu_train, y_edu_dev = train_test_split(
    X_edu, y_edu, test_size=0.15, random_state=42)

print(X_edu_train.shape)
print(X_edu_dev.shape)

(33514, 47)
(5915, 47)


In [30]:
bins = [0, 8, 12, 14, 17]
labels = [0, 1, 2, 3]  # 0=Low, 1=HS-level, 2=Some-college, 3=Advanced

y_edu_train_binned = pd.cut(y_edu_train, bins=bins, labels=labels).astype(int)
y_edu_dev_binned = pd.cut(y_edu_dev, bins=bins, labels=labels).astype(int)


In [34]:
#Model selection and prediction 

#Decision Tree
decision_tree= DecisionTreeClassifier(random_state=42, max_depth=10)

#fitting the classifier
decision_tree.fit(X_edu_train,y_edu_train_binned)

#making predictions
edu_y_prediction = decision_tree.predict(X_edu_dev)
edu_train_score = decision_tree.score(X_edu_train, y_edu_train_binned)
edu_dev_score = decision_tree.score(X_edu_dev, y_edu_dev_binned)
precision = precision_score(y_edu_dev_binned, edu_y_prediction,average='micro')
recall = recall_score(y_edu_dev_binned, edu_y_prediction, average='micro')

print(edu_y_prediction,edu_train_score,edu_dev_score, precision, recall)


[1 2 1 ... 1 1 1] 0.7416900399832905 0.7146238377007608 0.7146238377007608 0.7146238377007608
